---

Image datasets and measurement

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImageDatabaseWithManifest, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter, pglParameterBatch
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()

---

Load the image database

---

In [ ]:
# load the database of images. Will check in directory for image formats that PIL
# knows about and make a list. This does not load the images, or check to see if they are valid
#imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000")
imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",filenameColumn="image_filename",indexColumn="test_image_nr",captionColumn="concept")

---

Print and display images

---

In [ ]:
# display a single image
#imdb.images[111].display()

# print image metadata one-by-one, this may take some time because it 
# has to open each file 
#imdb.print() 
imdb.print()

---

Display image dataset in a dialog

---

In [ ]:
pgl.traitsDialog(imdb)

In [ ]:
img=imdb.getImage(0)

---

Make a task to display images

---

In [ ]:
class pglImageTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Image Task"
        self.settings.nTrials = 2
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
#            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",
#            'manifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"test_image_nr",'captionColumn':"concept"},
#            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000",
#            'manifestColumnNames': {'filenameColumn':"filename",'indexColumn':"index",'captionColumn':"caption_1"},
            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_catch_img_12reps",
            'manifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"catch_nr",'captionColumn':"original_filename"},
            'imdb': None,
            'nImages': 10,
            'imageSize': 18,
            'nImagesPerTrial': 10,
            'blankEvery': 45,
        }        
        p = self.settings.fixedParameters
        
        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5] * p['nImagesPerTrial']

        # initialize image database using parameters set from fixedParameters
        imdb = pglImageDatabaseWithManifest(
            p['imagesDirectory'],
            filenameColumn=p['manifestColumnNames']['filenameColumn'],
            indexColumn=p['manifestColumnNames']['indexColumn'],
            captionColumn=p['manifestColumnNames']['captionColumn'],
        )
        if imdb.nImages==0:
            pglMessages.warning(f"No images found in {p['imageDirectory']}")
        p['imdb'] = imdb
        
        # preload images
        for iImage in range(p['nImages']):
            imdb.preloadImage(iImage)
            
        # add parameter for image number
        imageNum = pglParameterBatch('imageNum',np.arange(p['nImages']),batchSize=p['nImagesPerTrial'], blankEvery=p['blankEvery'])
        self.addParameter(imageNum)
        
        # set current image
        self.state.currentImage = None

    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment % 2 == 0: 
            # get image database
            imdb = self.settings.fixedParameters['imdb']
            # get the current image number
            imageNums = self.currentParams['imageNum']
            if imageNums:
                # load an image
                imageNum = imageNums[int(self.state.currentSegment/2)]
                # get the image data
                img = imdb.getImage(imageNum)
                img.convert("RGB")
                print(f"img: {img}")
                # turn into a pglImage
                self.state.currentImage = self.pgl.imageCreate(np.array(img))
            else:
                self.state.currentImage = None
    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment % 2 == 0: 
            if self.state.currentImage:
                self.state.currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

        


---

Setup experiment

---

In [24]:
pgl.cleanUp()
#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='imageTask')

imageTask = pglImageTask(pgl)
e.addTask(imageTask)

(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serialization name 'pglImageTask' for pglImageTask and pglImageTask ⚠️
(pglMessages:warning) ⚠️ Duplicate serializati

---

run experiment

---

In [25]:
e.initScreen()
e.run()

(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
================================= pglBase:open =================================
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglBase:getMetalAppName) Using latest build: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Starting mglMetal application: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260814_210859.wliMp56BP9
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260814_210859.wliMp56BP9
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglKeyboardMouse:start) Starting keyboard and mouse event listener.
(pglEventListener) Eating 7 keys: ['1', '2', '3', '4', '`', 'escape', 

In [ ]:
e.tasks[0].settings.fixedParameters

In [ ]:
pgl.open(0)

In [ ]:
pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

pgl.flush()